# Аудит аплифта дорожных карт DD

## tl;dr

- Проверено 204 логических items / 181 numeric uplift; детерминированно сопоставлены 150 items (82,9%), 31 вынесен в «Не сопоставлено».
- Выявлено 45 flagged product/block/metric sets: Excel total 252,8 п.п., actual metric-set uplift 129,7 п.п., overstatement 123,1 п.п.
- Неидентичных пересекающихся sets не найдено (0); алгоритм всё равно исключает такие sets из error claims.
- Reviewed mappings: DP!4 → три метрики блока Цели (actual 10,6); СВР!7 и УБ!310 → обе метрики целевого рынка (1,6 / 2,1); DB!5 и PC!7 → Discovery >=40% бэклога.
- Страхование залога: УБ!305 = 2,9 vs 2,1; УБ!306 = 2,9 vs 2,1; УБ!302 = 2,9 vs 0,0 как excluded. Full-block 12,4 не используется.
- ДСЖ ПК: УБ!130 явно сопоставлен с «Клиенты с продуктом» + «MAU продукта», Excel 30,0 vs actual set 1,9 п.п.


## Context & Methods

### Key Assumptions

- Авторитетное зерно источника: 204 логических roadmap items, из них 181 с числовым uplift.
- Сопоставление разрешено только детерминированно: нормализованное exact/containment, проверенные block-specific regex-псевдонимы и явно документированные row-level решения.
- Каталог кандидатов включает все DD-метрики, в том числе исключённые, неприменимые и `dd_calculation_flg = 0`; их фактический uplift всегда равен 0.
- Знаменатель продукта — сумма `max_value` только включённых DD-метрик.
- Сравнение выполняется на зерне уникального набора метрик. Items с одинаковым набором объединяются до сравнения, поэтому cap не переиспользуется.
- Пересекающиеся неидентичные наборы исключаются из утверждений об ошибках и выносятся в «Не сопоставлено».
- Все вычисления выполняются через `Decimal`; итоговые сравнения используют `ROUND_HALF_UP` до 1 знака.


### 1. Setup and visible parameters

Для повторного запуска нужны Python 3.11 и `openpyxl==3.1.5`. Ноутбук создан и исполнен через `nbformat`/`nbclient`; исходные XLSX и JSON читаются без изменений.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import json
import math
import re
import unicodedata

from openpyxl import Workbook, load_workbook
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter, range_boundaries

AUDIT_AS_OF = date(2026, 8, 24)
SOURCE_XLSX_REL = Path("Дорожные карты по повышению рейтинга DD.xlsx")
SOURCE_JSON_REL = Path("gravity-app/public/report-data.json")
OUTPUT_XLSX_REL = Path("artifacts/Аудит_аплифта_дорожных_карт_DD.xlsx")

repo_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next(
    (path.resolve() for path in repo_candidates
     if (path / SOURCE_XLSX_REL).exists() and (path / SOURCE_JSON_REL).exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Не удалось найти корень репозитория с обоими источниками")

SOURCE_XLSX = REPO_ROOT / SOURCE_XLSX_REL
SOURCE_JSON = REPO_ROOT / SOURCE_JSON_REL
OUTPUT_XLSX = REPO_ROOT / OUTPUT_XLSX_REL
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)

ONE_DECIMAL = Decimal("0.1")
HUNDRED = Decimal("100")

print(f"Repository root: {REPO_ROOT}")
print(f"Roadmap source: {SOURCE_XLSX_REL}")
print(f"DD source: {SOURCE_JSON_REL}")
print(f"Output: {OUTPUT_XLSX_REL}")

Repository root: /Users/roman/Documents/Codex/DD-dev
Roadmap source: Дорожные карты по повышению рейтинга DD.xlsx
DD source: gravity-app/public/report-data.json
Output: artifacts/Аудит_аплифта_дорожных_карт_DD.xlsx


### 2. Deterministic normalization and reviewed mappings

Сохраняются существующие product/block aliases. Метрики ищутся только внутри канонического блока item: сначала по нормализованному exact/containment, затем по явному проверенному regex-словарю. Никакой fuzzy-выбор не используется.


In [2]:
def normalize_text(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_match_text(value) -> str:
    text = normalize_text(value).casefold().replace("ё", "е")
    return re.sub(r"[^0-9a-zа-я]+", " ", text).strip()


def as_decimal(value, *, allow_none: bool = False):
    if value is None or value == "":
        if allow_none:
            return None
        raise ValueError("Пустое число")
    if isinstance(value, bool):
        raise TypeError("Boolean не является числовым аплифтом")
    if isinstance(value, Decimal):
        return value
    if isinstance(value, (int, float)):
        if isinstance(value, float) and not math.isfinite(value):
            raise ValueError(f"Неконечное число: {value}")
        return Decimal(str(value))
    return Decimal(normalize_text(value).replace(",", "."))


def round_1(value: Decimal) -> Decimal:
    return value.quantize(ONE_DECIMAL, rounding=ROUND_HALF_UP)


UNIT_ALIASES = {"СВР": "CBP"}

PRODUCT_ALIASES_RAW = {
    ("CBP", "Вклады + НС"): "Вклады+НС",
    ("CBP", "ПК"): "Потребительский кредит",
    ("CX", "ПУ СберПремьер"): "Пакет услуг СберПремьер",
    ("CX", "ПУ СберПервый"): "Пакет услуг СберПервый",
    ("CX", "TA"): "Top Affluent",
    ("PC", "Выписки и справки"): "Выписки, справки",
    ("ДомКлик", "Сделка вторичка, Загородка"): "Сделка вторичка",
    ("ДомКлик", "Сделка ИЖС, Загородка"): "Сделка ИЖС",
}
PRODUCT_ALIASES = {
    (normalize_text(unit), normalize_text(source)): normalize_text(target)
    for (unit, source), target in PRODUCT_ALIASES_RAW.items()
}

BLOCK_ORDER = [
    "Знание ключевых метрик", "Цели", "Воронка привлечения", "Воронка оттока",
    "Алерты", "Механики", "Гипотезы и инициативы", "Клиентский опыт",
    "Воронка использования", "Воронка онбординга", "Воронка продаж",
    "Воронка входа в канал", "Данные",
]
BLOCK_ORDER_INDEX = {name: index for index, name in enumerate(BLOCK_ORDER)}

BLOCK_ALIASES_RAW = {
    "CX Score": ("Клиентский опыт",),
    "Цели уровня ЛЮ/ЛТ": ("Цели",),
    "Гипотезы": ("Гипотезы и инициативы",),
    "Знание ключевых метрик\nЦели": ("Знание ключевых метрик", "Цели"),
}
BLOCK_ALIASES = {
    normalize_text(source): tuple(targets)
    for source, targets in BLOCK_ALIASES_RAW.items()
}
KNOWN_BLOCK_BY_NORMALIZED = {normalize_text(name): name for name in BLOCK_ORDER}


def canonical_block_set(raw_block):
    normalized = normalize_text(raw_block)
    if not normalized:
        return None, "Пустой блок"
    if normalized in BLOCK_ALIASES:
        blocks = BLOCK_ALIASES[normalized]
    elif normalized in KNOWN_BLOCK_BY_NORMALIZED:
        blocks = (KNOWN_BLOCK_BY_NORMALIZED[normalized],)
    else:
        return None, "Блок не сопоставлен"
    return tuple(sorted(set(blocks), key=lambda name: BLOCK_ORDER_INDEX[name])), None


# (rule id, reviewed regex over normalized activity, explicitly named metric set)
METRIC_ALIAS_RULES = {
    "Знание ключевых метрик": [
        ("knowledge_tam_ru", r"объем целевого рынка(?: в)? росс", ("Объем целевого рынка в России",)),
        ("knowledge_tam_sber", r"объем целевого рынка(?: в)? сбер", ("Объем целевого рынка в Сбере",)),
        ("knowledge_clients", r"клиент[ыа-я ]* с продукт", ("Клиенты с продуктом",)),
        ("knowledge_mau", r"\bmau\b|\bмау\b", ("MAU продукта",)),
        ("knowledge_satellites", r"продукт[а-я ]*спутник", ("Знание продуктов спутников",)),
        ("knowledge_reporting", r"пониман[а-я ]*отчетност|гемба[а-я ]*отчетност|знани[а-я ]*отчетност|отчетност[а-я ]+и знан", ("Знание об отчетности в Навигаторе",)),
        ("knowledge_active_base", r"активн[а-я ]*клиентск[а-я ]*баз", ("Активная клиентская база",)),
        ("knowledge_tam_sam", r"воронк[а-я ]+tam sam", ("Объем целевого рынка в России", "Объем целевого рынка в Сбере")),
    ],
    "Цели": [
        ("goals_drivers", r"факторн[а-я ]*анализ|декомпоз[а-я ]*драйвер|выве[а-я ]+драйвер|драйвер[а-я ]*цел", ("Факторный анализ - драйверы 1-2 ур.",)),
        ("goals_forecast", r"прогноз", ("Прогноз по целям",)),
        ("goals_monitoring", r"цели выведен[а-я ]*(?:в|на)|выве[а-я ]+цели|обогат[а-я ]+целями|наличи[а-я ]*отчетов[а-я ]*навигатор", ("Цели выведены на мониторинг",)),
    ],
    "Воронка привлечения": [
        ("acq_reporting", r"настро[а-я ]*отчетност|настро[а-я ]*мониторинг|постро[а-я ]*воронк[а-я ]*оформлен", ("Настроена отчетность",)),
        ("acq_fullness", r"полнот[а-я ]*отчет|отчетност[а-я ]*обогащен[а-я ]*метрик|обогат[а-я ]*метрик[а-я ]*мониторинг|источник[а-я /]*cr|покрыти[а-я ]*канал", ("Полнота отчета",)),
        ("acq_regular", r"регулярност", ("Регулярность",)),
        ("acq_bench", r"бенчмарк", ("Наличие бенчмарков",)),
        ("acq_analysis", r"комплексн[а-я ]*анализ[а-я ]*воронк[а-я ]*привлеч|анализ[а-я ]*привлеч|оценк[а-я ]*эффективност[а-я ]*точк[а-я ]*рост", ("Проведение комплексного анализа воронки привлечения",)),
        ("acq_initiatives", r"переч[а-я ]*инициатив[а-я ]*привлеч", ("Составлен перечень инициатив по привлечению",)),
        ("acq_drafts", r"черновик", ("Черновики в СБОЛ >=70%",)),
        ("acq_self_service", r"наличи[а-я ]*self service|наличи[а-я ]*селф сервис", ("Наличие Self-service",)),
        ("acq_success", r"успешн[а-я ]*(?:бизнес )?запуск|успешн[а-я ]*пилот[а-я ]*(?:через|с помощью)[а-я ]*(?:ss|self service)", ("Наличие успешных бизнес-запусков",)),
        ("acq_launches", r"запуск[а-я ]*кампан[а-я ]*квартал", ("Запуски кампаний за квартал",)),
    ],
    "Воронка оттока": [
        ("churn_reporting", r"настро[а-я ]*отчетност|настро[а-я ]*мониторинг", ("Настроена отчетность",)),
        ("churn_fullness", r"полнот[а-я ]*отчет|обогат[а-я ]*метрик[а-я ]*мониторинг", ("Полнота отчета",)),
        ("churn_regular", r"регулярност", ("Регулярность",)),
        ("churn_bench", r"бенчмарк", ("Наличие бенчмарков",)),
        ("churn_analysis", r"комплексн[а-я ]*анализ[а-я ]*воронк[а-я ]*отток|анализ[а-я ]*отток|точк[а-я ]*рост", ("Проведение комплексного анализа воронки оттока",)),
        ("churn_actions", r"мероприят[а-я ]*снижени[а-я ]*отток|работ[а-я ]*с отклонен", ("Мероприятия по работе с отклонениями",)),
    ],
    "Алерты": [
        ("alerts_business", r"оповещен[а-я ]*бизнес метрик|светофор[а-я ]*цел|оповещен[а-я ]*ворон", ("Оповещения по бизнес-метрикам",)),
        ("alerts_system", r"системн[а-я ]*сбо", ("Оповещения по системным сбоям",)),
        ("alerts_ux", r"ux|ui|клиентск[а-я ]*опыт", ("Оповещения по проблемам, связанным с UX/UI",)),
    ],
    "Механики": [
        ("mechanics_return", r"реактивац|возврат[а-я ]*клиент", ("Возврат клиентов",)),
        ("mechanics_retention", r"удержан[а-я ]*клиент", ("Удержание клиентов",)),
        ("mechanics_flexible", r"гибк[а-я ]*изменен[а-я ]*услов", ("Гибкое изменение условий продукта",)),
        ("mechanics_cross_sell", r"cross sell|кросс sell|кросс селл", ("Cross-sell",)),
        ("mechanics_up_sell", r"допродаж|up sell", ("Допродажи (up-sell)",)),
        ("mechanics_monitoring", r"мониторинг[а-я ]*механик|ключев[а-я ]*механик", ("Мониторинг механик",)),
        ("mechanics_level", r"повышен[а-я ]*уровн[а-я ]*клиент", ("Повышение уровня клиента",)),
        ("mechanics_personalization", r"персонализац[а-я ]*клиентск[а-я ]*подсегмент|услови[а-я ]*персонализац", ("Персонализация",)),
    ],
    "Гипотезы и инициативы": [
        ("hyp_ab", r"\ba b тест|\bа b тест|\bab тест", ("A/B-тесты",)),
        ("hyp_discovery", r"discovery|доля[а-я ]*исследован|research задач|бэклог[а-я ]*аналитик[а-я ]*research|сфокусир[а-я ]*бэклог", ("Discovery >=40% бэклога",)),
        ("hyp_quality", r"оценк[а-я ]*исследован|качеств[а-я ]*(?:проведенн[а-я ]*)?исс?ледован|качеств[а-я ]*исседован", ("Оценка исследований >=7,5",)),
        ("hyp_extra", r"доп[а-я .]*инициатив[а-я ]*сверх|инициатив[а-я ]*сверх[а-я ]*бп", ("Доп. инициативы сверх БП",)),
    ],
    "Клиентский опыт": [
        ("cx_score", r"cx score", ("CX Score",)),
        ("ux_score", r"ux score", ("UХ Score",)),
    ],
}

for funnel_block in ("Воронка продаж", "Воронка использования", "Воронка онбординга", "Воронка входа в канал"):
    METRIC_ALIAS_RULES[funnel_block] = [
        ("funnel_reporting", r"настро[а-я ]*отчетност", ("Настроена отчетность",)),
        ("funnel_fullness", r"полнот[а-я ]*отчет|обогат[а-я ]*метрик", ("Полнота отчета",)),
        ("funnel_regular", r"регулярност", ("Регулярность",)),
        ("funnel_bench", r"бенчмарк", ("Наличие бенчмарков",)),
        ("funnel_analysis", r"комплексн[а-я ]*анализ|анализ[а-я ]*ворон", (f"Проведение комплексного анализа {funnel_block.casefold()}",)),
        ("funnel_actions", r"работ[а-я ]*с отклонен|переч[а-я ]*мероприят", ("Мероприятия по работе с отклонениями",)),
    ]

# Проверенные item-level решения устраняют известную ложную вложенность phrase matching.
REVIEWED_ITEM_METRIC_MAP = {
    ("DP", 4): {
        "product": "СберИнвестор",
        "metrics": (("Цели", "Цели выведены на мониторинг"), ("Цели", "Факторный анализ - драйверы 1-2 ур."), ("Цели", "Прогноз по целям")),
        "note": "Декомпозиция, прогноз и вывод в Навигатор → все три метрики блока Цели.",
    },
    ("СВР", 7): {
        "product": "ПК",
        "metrics": (("Знание ключевых метрик", "Объем целевого рынка в России"), ("Знание ключевых метрик", "Объем целевого рынка в Сбере")),
        "note": "Единая формулировка «объем целевого рынка России и Сбера» → обе target-market метрики.",
    },
    ("УБ", 310): {
        "product": "ЦФА",
        "metrics": (("Знание ключевых метрик", "Объем целевого рынка в России"), ("Знание ключевых метрик", "Объем целевого рынка в Сбере")),
        "note": "Единая формулировка «объем целевого рынка России и Сбера» → обе target-market метрики.",
    },
    ("DB", 5): {
        "product": "Оплата улыбкой",
        "metrics": (("Гипотезы и инициативы", "Discovery >=40% бэклога"),),
        "note": "Доля исследований >=40% → Discovery >=40% бэклога.",
    },
    ("PC", 7): {
        "product": "Выписки и справки",
        "metrics": (("Гипотезы и инициативы", "Discovery >=40% бэклога"),),
        "note": "Доля исследовательских задач >=40% → Discovery >=40% бэклога.",
    },
    ("УБ", 130): {
        "product": "ДСЖ ПК",
        "metrics": (("Знание ключевых метрик", "Клиенты с продуктом"), ("Знание ключевых метрик", "MAU продукта")),
        "note": "«Активные Клиенты & MAU Продуктов и Сервисов» → Клиенты с продуктом + MAU продукта.",
    },
    ("УБ", 302): {
        "product": "Страхование залога",
        "metrics": (("Воронка привлечения", "Регулярность"),),
        "note": "Явно названная Регулярность; метрика исключена из DD, поэтому actual=0.",
    },
    ("УБ", 305): {
        "product": "Страхование залога",
        "metrics": (("Воронка привлечения", "Наличие Self-service"),),
        "note": "Явно названо Наличие Self-service.",
    },
    ("УБ", 306): {
        "product": "Страхование залога",
        "metrics": (("Воронка привлечения", "Наличие успешных бизнес-запусков"),),
        "note": "Self-Service является контекстом; целевая метрика — Наличие успешных бизнес-запусков.",
    },
}

AMBIGUOUS_ALIAS_RULES = {
    "Алерты": [
        (r"систем[а-я ]*оповещен|внедр[а-я ]*оповещен", ("Оповещения по бизнес-метрикам", "Оповещения по системным сбоям", "Оповещения по проблемам, связанным с UX/UI")),
    ],
    "Воронка привлечения": [
        (r"кампейнинг|кампан", ("Запуски кампаний за квартал", "Наличие Self-service", "Наличие успешных бизнес-запусков")),
    ],
}

print("Unit aliases:", UNIT_ALIASES)
print("Product aliases:", PRODUCT_ALIASES_RAW)
print("Block aliases:", BLOCK_ALIASES_RAW)
print("Reviewed regex rules:", sum(len(rules) for rules in METRIC_ALIAS_RULES.values()))
print("Reviewed item-level mappings:", REVIEWED_ITEM_METRIC_MAP)


Unit aliases: {'СВР': 'CBP'}
Product aliases: {('CBP', 'Вклады + НС'): 'Вклады+НС', ('CBP', 'ПК'): 'Потребительский кредит', ('CX', 'ПУ СберПремьер'): 'Пакет услуг СберПремьер', ('CX', 'ПУ СберПервый'): 'Пакет услуг СберПервый', ('CX', 'TA'): 'Top Affluent', ('PC', 'Выписки и справки'): 'Выписки, справки', ('ДомКлик', 'Сделка вторичка, Загородка'): 'Сделка вторичка', ('ДомКлик', 'Сделка ИЖС, Загородка'): 'Сделка ИЖС'}
Block aliases: {'CX Score': ('Клиентский опыт',), 'Цели уровня ЛЮ/ЛТ': ('Цели',), 'Гипотезы': ('Гипотезы и инициативы',), 'Знание ключевых метрик\nЦели': ('Знание ключевых метрик', 'Цели')}
Reviewed regex rules: 68
Reviewed item-level mappings: {('DP', 4): {'product': 'СберИнвестор', 'metrics': (('Цели', 'Цели выведены на мониторинг'), ('Цели', 'Факторный анализ - драйверы 1-2 ур.'), ('Цели', 'Прогноз по целям')), 'note': 'Декомпозиция, прогноз и вывод в Навигатор → все три метрики блока Цели.'}, ('СВР', 7): {'product': 'ПК', 'metrics': (('Знание ключевых метрик', 'Объем 

## Data

### 3. Load and validate the complete DD metric catalog

Все метрики сохраняются как кандидаты. Inclusion-флаг вычисляется отдельно и влияет только на знаменатель и actual uplift.


In [3]:
with SOURCE_JSON.open("r", encoding="utf-8") as source_file:
    report_data = json.load(source_file, parse_float=Decimal, parse_int=Decimal)

products = report_data.get("products", [])
product_index = {}
duplicate_product_keys = []
all_metric_records = []
included_metric_records = []
null_included_values = []
value_above_max = []
negative_values = []


def inclusion_reasons(metric):
    reasons = []
    max_value = as_decimal(metric.get("max_value"), allow_none=True)
    if max_value is None or max_value <= 0:
        reasons.append("max_value отсутствует или <= 0")
    if metric.get("is_applicabble_flg") is False:
        reasons.append("is_applicabble_flg=false")
    if metric.get("excluded_from_index") is True:
        reasons.append("excluded_from_index=true")
    if metric.get("dd_calculation_flg") == 0:
        reasons.append("dd_calculation_flg=0")
    return reasons


def metric_is_included(metric):
    return not inclusion_reasons(metric)


for product in products:
    key = (normalize_text(product.get("unit")), normalize_text(product.get("name")))
    if key in product_index:
        duplicate_product_keys.append(key)
    product_index[key] = product
    for block in product.get("metrics", []):
        block_name = normalize_text(block.get("name"))
        for metric in block.get("metrics", []):
            max_value = as_decimal(metric.get("max_value"), allow_none=True)
            value = as_decimal(metric.get("value"), allow_none=True)
            included = metric_is_included(metric)
            record = {
                "unit": key[0], "product": key[1], "block": block_name,
                "metric": normalize_text(metric.get("name")), "value": value,
                "max_value": max_value, "included": included,
                "inclusion_reasons": inclusion_reasons(metric),
                "raw": metric,
            }
            all_metric_records.append(record)
            if not included:
                continue
            included_metric_records.append(record)
            if value is None:
                null_included_values.append(record)
            else:
                if value > max_value:
                    value_above_max.append(record)
                if value < 0:
                    negative_values.append(record)

assert not duplicate_product_keys, f"Дубли product×unit в JSON: {duplicate_product_keys}"
assert not null_included_values, f"У включённых метрик есть пустые value: {null_included_values[:3]}"

print({
    "json_products": len(products),
    "all_candidate_metrics": len(all_metric_records),
    "included_metrics": len(included_metric_records),
    "excluded_or_inapplicable_metrics": len(all_metric_records) - len(included_metric_records),
    "duplicate_product_keys": len(duplicate_product_keys),
    "null_included_values": len(null_included_values),
    "value_above_max": len(value_above_max),
    "negative_values": len(negative_values),
})


{'json_products': 80, 'all_candidate_metrics': 2949, 'included_metrics': 2550, 'excluded_or_inapplicable_metrics': 399, 'duplicate_product_keys': 0, 'null_included_values': 0, 'value_above_max': 0, 'negative_values': 0}


### 4. Read logical roadmap items with the application’s merged-cell semantics

Алгоритм полностью повторяет `build_calc_report.read_roadmap_workbook()`: распознаёт лист по заголовкам, создаёт item только на стартовой строке логического merge-диапазона и объединяет непустые физические строки E в одно `planned_activity`. Обычные пустые блоки остаются пустыми.

In [4]:
roadmap_formula_wb = load_workbook(SOURCE_XLSX, data_only=False, read_only=False)
roadmap_values_wb = load_workbook(SOURCE_XLSX, data_only=True, read_only=False)


def clean_roadmap_text(value) -> str:
    return "" if value is None else str(value).strip()


def normalize_lookup_key(value) -> str:
    return re.sub(r"\s+", " ", clean_roadmap_text(value)).casefold()


def normalize_roadmap_key(value) -> str:
    normalized = unicodedata.normalize("NFKC", clean_roadmap_text(value)).casefold().replace("ё", "е")
    return re.sub(r"[^0-9a-zа-я]+", "", normalized)


def roadmap_unit_name(sheet_name) -> str:
    source = clean_roadmap_text(sheet_name)
    for alias, canonical in UNIT_ALIASES.items():
        if normalize_roadmap_key(source) == normalize_roadmap_key(alias):
            return canonical
    return source


def roadmap_profile_name(unit: str, source_name) -> str:
    name = re.sub(r"\s+", " ", clean_roadmap_text(source_name))
    for (alias_unit, alias_name), canonical in PRODUCT_ALIASES_RAW.items():
        if (
            normalize_roadmap_key(unit) == normalize_roadmap_key(alias_unit)
            and normalize_roadmap_key(name) == normalize_roadmap_key(alias_name)
        ):
            return canonical
    return name


def roadmap_cell_value(worksheet, row: int, column: int):
    for merged_range in worksheet.merged_cells.ranges:
        if (
            merged_range.min_row <= row <= merged_range.max_row
            and merged_range.min_col <= column <= merged_range.max_col
        ):
            return worksheet.cell(merged_range.min_row, merged_range.min_col).value
    return worksheet.cell(row, column).value


def roadmap_merged_end_row(worksheet, row: int, column: int) -> int:
    for merged_range in worksheet.merged_cells.ranges:
        if (
            merged_range.min_row <= row <= merged_range.max_row
            and merged_range.min_col <= column <= merged_range.max_col
        ):
            return merged_range.max_row
    return row


def roadmap_merged_start_row(worksheet, row: int, column: int) -> int:
    for merged_range in worksheet.merged_cells.ranges:
        if (
            merged_range.min_row <= row <= merged_range.max_row
            and merged_range.min_col <= column <= merged_range.max_col
        ):
            return merged_range.min_row
    return row


def is_roadmap_sheet(worksheet) -> bool:
    return (
        normalize_lookup_key(worksheet.cell(1, 2).value).startswith("продукт")
        and normalize_lookup_key(worksheet.cell(1, 3).value) == "квартал"
        and normalize_lookup_key(worksheet.cell(1, 4).value) == "блок для развития"
        and normalize_lookup_key(worksheet.cell(1, 5).value) == "планируемое мероприятие"
        and normalize_lookup_key(worksheet.cell(1, 7).value).startswith("ожидание прироста индекса dd, п.п")
    )


roadmap_items = []
recognized_sheets = 0
source_numeric_uplifts = 0
source_formula_uplifts = 0

for ws_values in roadmap_values_wb.worksheets:
    if not is_roadmap_sheet(ws_values):
        continue
    recognized_sheets += 1
    ws_formula = roadmap_formula_wb[ws_values.title]
    unit = roadmap_unit_name(ws_values.title)

    for source_row in range(2, ws_values.max_row + 1):
        raw_uplift = ws_values.cell(source_row, 7).value
        has_uplift = raw_uplift is not None and clean_roadmap_text(raw_uplift) != ""
        uplift_decimal = None
        if has_uplift:
            try:
                uplift_decimal = as_decimal(raw_uplift)
            except (TypeError, ValueError, ArithmeticError) as error:
                raise ValueError(
                    f"Некорректный ожидаемый прирост в {ws_values.title}!G{source_row}: {raw_uplift!r}"
                ) from error
            source_numeric_uplifts += 1
            formula_value = ws_formula.cell(source_row, 7).value
            if isinstance(formula_value, str) and formula_value.startswith("="):
                source_formula_uplifts += 1
        else:
            raw_activity = ws_values.cell(source_row, 5).value
            if not isinstance(raw_activity, str) or not clean_roadmap_text(raw_activity):
                continue
            logical_columns = (4, 6, 8)
            if any(
                roadmap_merged_start_row(ws_values, source_row, column) != source_row
                for column in logical_columns
            ):
                continue

        source_name = clean_roadmap_text(roadmap_cell_value(ws_values, source_row, 2))
        if not source_name:
            raise ValueError(
                f"Не указана команда/продукт для ожидаемого прироста в {ws_values.title}!G{source_row}"
            )
        if normalize_lookup_key(source_name).startswith("продукт"):
            continue

        merged_end_row = (
            roadmap_merged_end_row(ws_values, source_row, 7)
            if has_uplift
            else max(
                roadmap_merged_end_row(ws_values, source_row, column)
                for column in (4, 6, 8)
            )
        )
        activity_parts = [
            activity
            for activity_row in range(source_row, merged_end_row + 1)
            if (activity := clean_roadmap_text(ws_values.cell(activity_row, 5).value))
        ]
        profile_name = roadmap_profile_name(unit, source_name)
        product_json_name = PRODUCT_ALIASES.get(
            (normalize_text(unit), normalize_text(source_name)),
            normalize_text(profile_name),
        )
        roadmap_items.append({
            "unit_source": unit,
            "unit_json": normalize_text(unit),
            "product_source": source_name,
            "product_json_name": product_json_name,
            "block_source": clean_roadmap_text(roadmap_cell_value(ws_values, source_row, 4)),
            "activity": "\n".join(activity_parts),
            "uplift": uplift_decimal,
            "uplift_formula": (
                ws_formula.cell(source_row, 7).value
                if isinstance(ws_formula.cell(source_row, 7).value, str)
                and ws_formula.cell(source_row, 7).value.startswith("=")
                else None
            ),
            "source_sheet": ws_values.title,
            "source_row": source_row,
        })

assert recognized_sheets == 8, f"Ожидалось 8 roadmap-листов, найдено {recognized_sheets}"
assert len(roadmap_items) == 204, f"Нарушено авторитетное зерно: {len(roadmap_items)} вместо 204"
assert source_numeric_uplifts == 181, (
    f"Нарушено число item с numeric uplift: {source_numeric_uplifts} вместо 181"
)

print({
    "source_items": len(roadmap_items),
    "items_with_numeric_uplift": source_numeric_uplifts,
    "items_with_formula_uplift": source_formula_uplifts,
    "recognized_sheets": recognized_sheets,
    "sheets_with_items": sorted({item["source_sheet"] for item in roadmap_items}),
    "grain_check": "PASS (204 / 181)",
})

{'source_items': 204, 'items_with_numeric_uplift': 181, 'items_with_formula_uplift': 1, 'recognized_sheets': 8, 'sheets_with_items': ['CX', 'DB', 'DP', 'PC', 'ДомКлик', 'СВР', 'УБ'], 'grain_check': 'PASS (204 / 181)'}


### 5. Map logical items to specific metric sets and calculate caps

Формула набора: `100 × Σ(max(max_value − value, 0)) / знаменатель продукта` только для включённых matched metrics. Исключённая matched metric остаётся в сопоставлении, но даёт 0.


In [5]:
def product_metric_catalog(product):
    catalog = {}
    denominator = Decimal("0")
    for block in product.get("metrics", []):
        block_name = normalize_text(block.get("name"))
        for metric in block.get("metrics", []):
            metric_name = normalize_text(metric.get("name"))
            included = metric_is_included(metric)
            max_value = as_decimal(metric.get("max_value"), allow_none=True)
            value = as_decimal(metric.get("value"), allow_none=True)
            record = {
                "block": block_name,
                "metric": metric_name,
                "value": value,
                "max_value": max_value,
                "included": included,
                "inclusion_reasons": inclusion_reasons(metric),
            }
            catalog[(block_name, metric_name)] = record
            if included:
                denominator += max_value
    return denominator, catalog


product_catalogs = {key: product_metric_catalog(product) for key, product in product_index.items()}
zero_denominator_products = [key for key, (denominator, _) in product_catalogs.items() if denominator <= 0]
assert not zero_denominator_products, f"Нулевые знаменатели: {zero_denominator_products}"


def relevant_catalog(product_key, block_set):
    denominator, catalog = product_catalogs[product_key]
    return {
        key: record for key, record in catalog.items()
        if record["block"] in block_set
    }


def ordered_metric_keys(keys):
    return tuple(sorted(keys, key=lambda key: (BLOCK_ORDER_INDEX.get(key[0], 999), key[1])))


def best_candidates_for_unmatched(activity, block_set, candidates):
    normalized_activity = normalize_match_text(activity)
    reviewed = []
    for block in block_set or ():
        for pattern, names in AMBIGUOUS_ALIAS_RULES.get(block, []):
            if re.search(pattern, normalized_activity):
                reviewed.extend((block, name) for name in names if (block, name) in candidates)
    if reviewed:
        return ordered_metric_keys(set(reviewed))
    return ordered_metric_keys(candidates.keys())


def map_item_to_metrics(item):
    block_set, block_reason = canonical_block_set(item["block_source"])
    if block_reason:
        return None, block_reason, (), None

    product_key = (item["unit_json"], item["product_json_name"])
    if product_key not in product_index:
        return None, "Продукт не найден в JSON", (), None

    candidates = relevant_catalog(product_key, block_set)
    if not candidates:
        return None, "В каноническом блоке продукта нет DD-метрик", (), None

    reviewed_key = (item["source_sheet"], item["source_row"])
    if reviewed_key in REVIEWED_ITEM_METRIC_MAP:
        reviewed = REVIEWED_ITEM_METRIC_MAP[reviewed_key]
        assert item["product_source"] == reviewed["product"], (reviewed_key, item["product_source"])
        target_set = ordered_metric_keys(reviewed["metrics"])
        absent = [key for key in target_set if key not in candidates]
        assert not absent, f"Reviewed mapping указывает на отсутствующие метрики: {reviewed_key}, {absent}"
        return target_set, None, target_set, "reviewed_item"

    activity_norm = normalize_match_text(item["activity"])
    source_block_norm = normalize_match_text(item["block_source"])
    matches = set()
    match_methods = set()

    for metric_key in candidates:
        metric_norm = normalize_match_text(metric_key[1])
        if metric_norm and (metric_norm == activity_norm or metric_norm in activity_norm):
            matches.add(metric_key)
            match_methods.add("normalized_exact_or_containment")
        if source_block_norm and source_block_norm == metric_norm:
            matches.add(metric_key)
            match_methods.add("block_label_exact_metric")

    for block in block_set:
        for rule_id, pattern, metric_names in METRIC_ALIAS_RULES.get(block, []):
            if re.search(pattern, activity_norm):
                for metric_name in metric_names:
                    metric_key = (block, metric_name)
                    if metric_key in candidates:
                        matches.add(metric_key)
                        match_methods.add(f"reviewed_regex:{rule_id}")

    if not matches:
        best = best_candidates_for_unmatched(item["activity"], block_set, candidates)
        reason = "Общая или неоднозначная формулировка; детерминированного совпадения с конкретной метрикой нет"
        return None, reason, best, None

    return ordered_metric_keys(matches), None, ordered_metric_keys(matches), "; ".join(sorted(match_methods))


numeric_items = [item for item in roadmap_items if item["uplift"] is not None]
mapped_numeric_items = []
unmatched_numeric_items = []

for item in numeric_items:
    metric_set, reason, candidates, match_method = map_item_to_metrics(item)
    enriched = dict(item)
    enriched["metric_set"] = metric_set
    enriched["match_method"] = match_method
    enriched["best_candidates"] = candidates
    if reason:
        enriched["unmatched_reason"] = reason
        unmatched_numeric_items.append(enriched)
    else:
        mapped_numeric_items.append(enriched)

assert len(numeric_items) == 181
assert len(mapped_numeric_items) + len(unmatched_numeric_items) == 181
assert all(item["match_method"] != "single_available_candidate" for item in mapped_numeric_items)

generic_alert_rows = {
    (item["source_sheet"], item["source_row"]): item
    for item in unmatched_numeric_items
    if item["source_sheet"] == "CX" and item["source_row"] in {11, 17, 21, 25}
}
assert set(generic_alert_rows) == {("CX", 11), ("CX", 17), ("CX", 21), ("CX", 25)}
for item in generic_alert_rows.values():
    assert normalize_text(item["activity"]) == "Внедрить систему оповещений"
    assert item["unmatched_reason"]
    assert item["best_candidates"]


grouped_items = defaultdict(list)
for item in mapped_numeric_items:
    group_key = (
        item["unit_source"], item["product_source"], item["unit_json"],
        item["product_json_name"], item["metric_set"],
    )
    grouped_items[group_key].append(item)


group_results = []
for group_key, items in grouped_items.items():
    unit_source, product_source, unit_json, product_json_name, metric_set = group_key
    denominator, catalog = product_catalogs[(unit_json, product_json_name)]
    metric_rows = []
    for metric_key in metric_set:
        record = catalog[metric_key]
        if record["included"]:
            gap = max(record["max_value"] - record["value"], Decimal("0"))
            actual_raw = HUNDRED * gap / denominator
        else:
            gap = Decimal("0")
            actual_raw = Decimal("0")
        metric_rows.append({
            **record,
            "gap": gap,
            "actual_raw": actual_raw,
            "actual_rounded": round_1(actual_raw),
            "inclusion_status": (
                "Включена"
                if record["included"]
                else "Исключена: " + "; ".join(record["inclusion_reasons"])
            ),
        })
    excel_total_raw = sum((item["uplift"] for item in items), Decimal("0"))
    actual_set_raw = sum((row["actual_raw"] for row in metric_rows), Decimal("0"))
    group_results.append({
        "unit_source": unit_source,
        "product_source": product_source,
        "unit_json": unit_json,
        "product_json_name": product_json_name,
        "metric_set": metric_set,
        "items": items,
        "metric_rows": metric_rows,
        "excel_total_raw": excel_total_raw,
        "excel_total_rounded": round_1(excel_total_raw),
        "actual_set_raw": actual_set_raw,
        "actual_set_rounded": round_1(actual_set_raw),
        "denominator": denominator,
        "overlap": False,
        "overlap_with": set(),
    })

# Неидентичные пересекающиеся sets внутри продукта нельзя превращать в claims об ошибке.
by_product = defaultdict(list)
for group in group_results:
    by_product[(group["unit_json"], group["product_json_name"])].append(group)

for product_groups in by_product.values():
    for index, left in enumerate(product_groups):
        left_set = set(left["metric_set"])
        for right in product_groups[index + 1:]:
            right_set = set(right["metric_set"])
            if left_set != right_set and left_set.intersection(right_set):
                left["overlap"] = True
                right["overlap"] = True
                left["overlap_with"].add(right["metric_set"])
                right["overlap_with"].add(left["metric_set"])

for group in group_results:
    group["overstatement"] = group["excel_total_rounded"] - group["actual_set_rounded"]
    group["flagged"] = (
        not group["overlap"]
        and group["excel_total_rounded"] > group["actual_set_rounded"]
    )

group_results.sort(key=lambda g: (
    g["unit_source"], g["product_source"],
    tuple((BLOCK_ORDER_INDEX.get(block, 999), metric) for block, metric in g["metric_set"]),
))
flagged_groups = [group for group in group_results if group["flagged"]]
overlap_groups = [group for group in group_results if group["overlap"]]
overlap_item_count = sum(len(group["items"]) for group in overlap_groups)

print({
    "numeric_items": len(numeric_items),
    "mapped_numeric_items": len(mapped_numeric_items),
    "unmatched_numeric_items": len(unmatched_numeric_items),
    "unique_metric_set_groups": len(group_results),
    "overlap_groups_excluded_from_claims": len(overlap_groups),
    "overlap_items_excluded_from_claims": overlap_item_count,
    "flagged_metric_sets": len(flagged_groups),
})
print("Unmatched reasons:", dict(Counter(item["unmatched_reason"] for item in unmatched_numeric_items)))


{'numeric_items': 181, 'mapped_numeric_items': 150, 'unmatched_numeric_items': 31, 'unique_metric_set_groups': 150, 'overlap_groups_excluded_from_claims': 0, 'overlap_items_excluded_from_claims': 0, 'flagged_metric_sets': 45}
Unmatched reasons: {'Общая или неоднозначная формулировка; детерминированного совпадения с конкретной метрикой нет': 28, 'Пустой блок': 3}


## Results

### 6. Coverage, flagged totals, overlap handling and mandatory spot-checks


In [6]:
flagged_excel_sum = sum((group["excel_total_rounded"] for group in flagged_groups), Decimal("0"))
flagged_actual_sum = sum((group["actual_set_rounded"] for group in flagged_groups), Decimal("0"))
flagged_overstatement_sum = sum((group["overstatement"] for group in flagged_groups), Decimal("0"))
flagged_detail_metric_rows = sum(len(group["metric_rows"]) for group in flagged_groups)
flagged_source_item_count = sum(len(group["items"]) for group in flagged_groups)

summary = {
    "source_logical_items": len(roadmap_items),
    "source_numeric_uplift_items": len(numeric_items),
    "mapped_numeric_items": len(mapped_numeric_items),
    "mapping_coverage_pct": float(round_1(HUNDRED * Decimal(len(mapped_numeric_items)) / Decimal(len(numeric_items)))),
    "unmatched_numeric_items": len(unmatched_numeric_items),
    "metric_set_groups": len(group_results),
    "overlap_groups_excluded_from_claims": len(overlap_groups),
    "overlap_items_excluded_from_claims": overlap_item_count,
    "flagged_metric_sets": len(flagged_groups),
    "flagged_source_items": flagged_source_item_count,
    "flagged_detail_metric_rows": flagged_detail_metric_rows,
    "flagged_excel_sum_pp": float(flagged_excel_sum),
    "flagged_actual_metric_set_sum_pp": float(flagged_actual_sum),
    "flagged_overstatement_sum_pp": float(flagged_overstatement_sum),
}
print(summary)


def find_source_mapping(sheet, row):
    matches = [
        (group, item)
        for group in group_results
        for item in group["items"]
        if item["source_sheet"] == sheet and item["source_row"] == row
    ]
    assert len(matches) == 1, (sheet, row, len(matches))
    return matches[0]


reviewed_regression_spots = {
    ("DP", 4): ("СберИнвестор", (("Цели", "Цели выведены на мониторинг"), ("Цели", "Факторный анализ - драйверы 1-2 ур."), ("Цели", "Прогноз по целям")), Decimal("10.5"), Decimal("10.6")),
    ("СВР", 7): ("ПК", (("Знание ключевых метрик", "Объем целевого рынка в России"), ("Знание ключевых метрик", "Объем целевого рынка в Сбере")), Decimal("2.4"), Decimal("1.6")),
    ("УБ", 310): ("ЦФА", (("Знание ключевых метрик", "Объем целевого рынка в России"), ("Знание ключевых метрик", "Объем целевого рынка в Сбере")), Decimal("4.5"), Decimal("2.1")),
    ("DB", 5): ("Оплата улыбкой", (("Гипотезы и инициативы", "Discovery >=40% бэклога"),), Decimal("2.5"), Decimal("2.5")),
    ("PC", 7): ("Выписки и справки", (("Гипотезы и инициативы", "Discovery >=40% бэклога"),), Decimal("6.1"), Decimal("6.1")),
}
reviewed_regression_results = {}
for (sheet, row), (product, metrics, expected_excel, expected_actual) in reviewed_regression_spots.items():
    group, item = find_source_mapping(sheet, row)
    assert item["product_source"] == product
    assert item["match_method"] == "reviewed_item"
    assert group["metric_set"] == ordered_metric_keys(metrics)
    assert item["uplift"] == expected_excel
    assert group["actual_set_rounded"] == expected_actual
    reviewed_regression_results[f"{sheet}!{row}"] = {
        "metrics": [metric for _, metric in group["metric_set"]],
        "excel_pp": float(item["uplift"]),
        "actual_pp": float(group["actual_set_rounded"]),
    }
print("Reviewed regression spot-checks:", reviewed_regression_results)


mandatory_spots = {
    302: ("Регулярность", Decimal("2.9"), Decimal("0.0")),
    305: ("Наличие Self-service", Decimal("2.9"), Decimal("2.1")),
    306: ("Наличие успешных бизнес-запусков", Decimal("2.9"), Decimal("2.1")),
}
spot_results = {}
for source_row, (expected_metric, expected_excel, expected_actual) in mandatory_spots.items():
    group, item = find_source_mapping("УБ", source_row)
    assert item["product_source"] == "Страхование залога"
    assert group["excel_total_rounded"] == expected_excel
    assert len(group["metric_rows"]) == 1
    metric_row = group["metric_rows"][0]
    assert metric_row["metric"] == expected_metric
    assert metric_row["actual_rounded"] == expected_actual
    assert group["actual_set_rounded"] == expected_actual
    assert group["actual_set_rounded"] != Decimal("12.4")
    assert group["flagged"]
    spot_results[source_row] = {
        "metric": metric_row["metric"],
        "excel_pp": float(group["excel_total_rounded"]),
        "actual_pp": float(group["actual_set_rounded"]),
        "inclusion": metric_row["inclusion_status"],
    }
assert spot_results[302]["inclusion"].startswith("Исключена")
print("Mandatory Страхование залога spot-checks:", spot_results)

dsj_group, dsj_item = find_source_mapping("УБ", 130)
expected_dsj_set = (
    ("Знание ключевых метрик", "MAU продукта"),
    ("Знание ключевых метрик", "Клиенты с продуктом"),
)
assert dsj_group["metric_set"] == expected_dsj_set
assert dsj_group["excel_total_rounded"] == Decimal("30.0")
assert dsj_group["actual_set_rounded"] == Decimal("1.9")
assert dsj_group["flagged"]
print({
    "spot_check": "ДСЖ ПК / explicitly mapped metric set",
    "mapping": [metric for _, metric in dsj_group["metric_set"]],
    "excel_pp": float(dsj_group["excel_total_rounded"]),
    "actual_metric_set_pp": float(dsj_group["actual_set_rounded"]),
    "status": "PASS",
})

print("Топ-10 флагов по завышению:")
for group in sorted(flagged_groups, key=lambda g: g["overstatement"], reverse=True)[:10]:
    print(
        group["unit_source"], "|", group["product_source"], "|",
        " + ".join(metric for _, metric in group["metric_set"]),
        "| Excel", group["excel_total_rounded"],
        "| actual", group["actual_set_rounded"],
        "| overstatement", group["overstatement"],
    )


{'source_logical_items': 204, 'source_numeric_uplift_items': 181, 'mapped_numeric_items': 150, 'mapping_coverage_pct': 82.9, 'unmatched_numeric_items': 31, 'metric_set_groups': 150, 'overlap_groups_excluded_from_claims': 0, 'overlap_items_excluded_from_claims': 0, 'flagged_metric_sets': 45, 'flagged_source_items': 45, 'flagged_detail_metric_rows': 65, 'flagged_excel_sum_pp': 252.8, 'flagged_actual_metric_set_sum_pp': 129.7, 'flagged_overstatement_sum_pp': 123.1}
Reviewed regression spot-checks: {'DP!4': {'metrics': ['Прогноз по целям', 'Факторный анализ - драйверы 1-2 ур.', 'Цели выведены на мониторинг'], 'excel_pp': 10.5, 'actual_pp': 10.6}, 'СВР!7': {'metrics': ['Объем целевого рынка в России', 'Объем целевого рынка в Сбере'], 'excel_pp': 2.4, 'actual_pp': 1.6}, 'УБ!310': {'metrics': ['Объем целевого рынка в России', 'Объем целевого рынка в Сбере'], 'excel_pp': 4.5, 'actual_pp': 2.1}, 'DB!5': {'metrics': ['Discovery >=40% бэклога'], 'excel_pp': 2.5, 'actual_pp': 2.5}, 'PC!7': {'metri

### 7. Build the stakeholder workbook

Workbook остаётся прямоугольными Excel tables с фильтрами. Детализация содержит одну строку на метрику внутри flagged metric set; идентичные sets предварительно объединены.


In [7]:
from openpyxl.worksheet.table import Table, TableStyleInfo

DETAIL_HEADERS = [
    "Продукт",
    "Блок",
    "Метрика",
    "Планируемое мероприятие",
    "Аплифт из Excel, п.п.",
    "Настоящий аплифт метрики, п.п.",
    "Excel total группы, п.п.",
    "Настоящий аплифт набора метрик, п.п.",
    "Завышение, п.п.",
    "Количество объединённых items",
    "Листы / строки источника",
    "Знаменатель продукта",
    "Текущее значение метрики",
    "Максимум метрики",
    "Включение в DD",
    "Метод сопоставления",
    "Юнит",
    "Продукт в JSON",
]

SUMMARY_HEADERS = [
    "Продукт",
    "Блок",
    "Набор метрик",
    "Excel total группы, п.п.",
    "Настоящий аплифт набора метрик, п.п.",
    "Завышение, п.п.",
    "Количество объединённых items",
    "Планируемые мероприятия",
    "Листы / строки источника",
    "Знаменатель продукта",
    "Включение метрик",
    "Юнит",
    "Продукт в JSON",
]

UNMAPPED_HEADERS = [
    "Продукт",
    "Блок",
    "Планируемое мероприятие",
    "Аплифт из Excel, п.п.",
    "Причина",
    "Лучшие кандидаты",
    "Тип исключения",
    "Лист-источник",
    "Строка-источник",
    "Юнит",
    "Продукт в JSON",
]

METHOD_ROWS = [
    ("Параметр", "Значение"),
    ("Дата среза / аудита", AUDIT_AS_OF.isoformat()),
    ("Источник дорожных карт", str(SOURCE_XLSX_REL)),
    ("Источник DD-метрик", str(SOURCE_JSON_REL)),
    ("Выходной файл", str(OUTPUT_XLSX_REL)),
    ("Единица дорожной карты", "Логический item по merged-cell semantics build_calc_report.read_roadmap_workbook(); 204 items, из них 181 с numeric uplift."),
    ("Зерно аудита", "Логический roadmap item → конкретный product/block/metric set; идентичные наборы внутри продукта объединяются до сравнения."),
    ("Каталог кандидатов", "Все DD-метрики соответствующего product/block, включая excluded, inapplicable и dd_calculation_flg=0."),
    ("Детерминированное сопоставление", f"Нормализованный exact/containment имени метрики + явный проверенный block-specific phrase/regex alias map + {len(REVIEWED_ITEM_METRIC_MAP)} reviewed item-level решений. Fuzzy guesses запрещены."),
    ("Общие / неоднозначные формулировки", "Не получают full-block capacity; выводятся на лист «Не сопоставлено» с причиной и допустимыми кандидатами."),
    ("Включённая метрика", "max_value > 0; is_applicabble_flg не равно false; excluded_from_index не равно true; dd_calculation_flg не равно 0."),
    ("Знаменатель продукта", "Сумма max_value всех включённых DD-метрик продукта."),
    ("Фактический uplift набора", "100 × Σ(max(max_value − value, 0)) включённых matched metrics / знаменатель продукта. Исключённая matched metric даёт 0."),
    ("Округление", "Decimal ROUND_HALF_UP до 1 десятичного знака для Excel total, actual metric set и overstatement."),
    ("Группировка дублей", "Items, сопоставленные одному и тому же metric set в product, суммируются; cap применяется один раз."),
    ("Пересечения", f"{len(overlap_groups)} неидентичных пересекающихся metric sets ({overlap_item_count} source items) исключены из error claims и перечислены в «Не сопоставлено»."),
    ("Псевдонимы блоков", "CX Score → Клиентский опыт; Цели уровня ЛЮ/ЛТ → Цели; Гипотезы → Гипотезы и инициативы; многострочный «Знание ключевых метрик + Цели» → два блока."),
    ("Псевдонимы юнитов", "; ".join(f"{source} → {target}" for source, target in UNIT_ALIASES.items())),
    ("Псевдонимы продуктов", "; ".join(f"{unit}: {source} → {target}" for (unit, source), target in PRODUCT_ALIASES_RAW.items())),
    ("Reviewed mapping Цели", "DP!4 → Цели выведены на мониторинг + Факторный анализ - драйверы 1-2 ур. + Прогноз по целям; actual metric-set 10,6 п.п."),
    ("Reviewed mapping целевого рынка", "СВР!7 и УБ!310 → Объем целевого рынка в России + Объем целевого рынка в Сбере; actual metric-set 1,6 и 2,1 п.п. соответственно."),
    ("Reviewed mapping Discovery", "DB!5 и PC!7 → Discovery >=40% бэклога."),
    ("Reviewed mapping ДСЖ ПК", "УБ!130: «Активные Клиенты & MAU Продуктов и Сервисов» → Клиенты с продуктом + MAU продукта; Excel 30,0 п.п., actual metric-set 1,9 п.п. Это не ёмкость всего блока."),
    ("Reviewed mapping Страхование залога", "УБ!302 → Регулярность (excluded, actual 0,0); УБ!305 → Наличие Self-service (2,1); УБ!306 → Наличие успешных бизнес-запусков (2,1). Значение full-block 12,4 не используется."),
    ("Покрытие", f"{len(mapped_numeric_items)} из {len(numeric_items)} numeric items сопоставлены; {len(unmatched_numeric_items)} не сопоставлены."),
    ("Флаги", f"{len(flagged_groups)} product/block/metric sets; Excel total {flagged_excel_sum} п.п.; actual {flagged_actual_sum} п.п.; overstatement {flagged_overstatement_sum} п.п."),
    ("Оговорка по суммам", "Суммы — арифметика flagged metric sets и не являются портфельным потенциалом."),
]


def decimal_float(value):
    return None if value is None else float(value)


def format_metric_key(metric_key):
    return f"{metric_key[0]} / {metric_key[1]}"


def source_refs(items):
    return "; ".join(f'{item["source_sheet"]}!{item["source_row"]}' for item in items)


def joined_activities(items):
    return "\n---\n".join(item["activity"] for item in items)


workbook = Workbook()
workbook.remove(workbook.active)

ws_detail = workbook.create_sheet("Ошибки — детализация")
ws_detail.append(DETAIL_HEADERS)
for group in flagged_groups:
    refs = source_refs(group["items"])
    activities = joined_activities(group["items"])
    blocks = " + ".join(dict.fromkeys(block for block, _ in group["metric_set"]))
    methods = "; ".join(sorted({item["match_method"] for item in group["items"]}))
    for metric_row in group["metric_rows"]:
        ws_detail.append([
            group["product_source"],
            blocks,
            metric_row["metric"],
            activities,
            decimal_float(group["excel_total_rounded"]),
            decimal_float(metric_row["actual_rounded"]),
            decimal_float(group["excel_total_rounded"]),
            decimal_float(group["actual_set_rounded"]),
            decimal_float(group["overstatement"]),
            len(group["items"]),
            refs,
            decimal_float(group["denominator"]),
            decimal_float(metric_row["value"]),
            decimal_float(metric_row["max_value"]),
            metric_row["inclusion_status"],
            methods,
            group["unit_source"],
            group["product_json_name"],
        ])

ws_summary = workbook.create_sheet("Ошибки — по метрикам")
ws_summary.append(SUMMARY_HEADERS)
for group in flagged_groups:
    blocks = " + ".join(dict.fromkeys(block for block, _ in group["metric_set"]))
    ws_summary.append([
        group["product_source"],
        blocks,
        " | ".join(format_metric_key(metric_key) for metric_key in group["metric_set"]),
        decimal_float(group["excel_total_rounded"]),
        decimal_float(group["actual_set_rounded"]),
        decimal_float(group["overstatement"]),
        len(group["items"]),
        joined_activities(group["items"]),
        source_refs(group["items"]),
        decimal_float(group["denominator"]),
        " | ".join(row["inclusion_status"] for row in group["metric_rows"]),
        group["unit_source"],
        group["product_json_name"],
    ])

ws_unmapped = workbook.create_sheet("Не сопоставлено")
ws_unmapped.append(UNMAPPED_HEADERS)
for item in sorted(unmatched_numeric_items, key=lambda x: (x["source_sheet"], x["source_row"])):
    candidate_text = " | ".join(format_metric_key(key) for key in item["best_candidates"])
    ws_unmapped.append([
        item["product_source"], item["block_source"], item["activity"],
        decimal_float(item["uplift"]), item["unmatched_reason"], candidate_text,
        "Несопоставленный numeric item", item["source_sheet"], item["source_row"],
        item["unit_source"], item["product_json_name"],
    ])

for group in overlap_groups:
    overlap_text = " || ".join(
        " | ".join(format_metric_key(key) for key in metric_set)
        for metric_set in sorted(group["overlap_with"])
    )
    ws_unmapped.append([
        group["product_source"],
        " + ".join(dict.fromkeys(block for block, _ in group["metric_set"])),
        joined_activities(group["items"]),
        decimal_float(group["excel_total_rounded"]),
        "Неидентичный пересекающийся metric set; исключён из error claims",
        overlap_text,
        "Пересечение metric sets",
        group["items"][0]["source_sheet"],
        source_refs(group["items"]),
        group["unit_source"],
        group["product_json_name"],
    ])

ws_method = workbook.create_sheet("Методика")
for row in METHOD_ROWS:
    ws_method.append(row)

header_fill = PatternFill("solid", fgColor="0B6B50")
header_font = Font(color="FFFFFF", bold=True)
red_fill = PatternFill("solid", fgColor="FFC7CE")
thin_gray = Side(style="thin", color="D9E2E3")
border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

for ws in workbook.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    ws.row_dimensions[1].height = 34
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = border

numeric_format_headers = {
    ws_detail: {
        "Аплифт из Excel, п.п.", "Настоящий аплифт метрики, п.п.",
        "Excel total группы, п.п.", "Настоящий аплифт набора метрик, п.п.",
        "Завышение, п.п.", "Знаменатель продукта", "Текущее значение метрики",
        "Максимум метрики",
    },
    ws_summary: {
        "Excel total группы, п.п.", "Настоящий аплифт набора метрик, п.п.",
        "Завышение, п.п.", "Знаменатель продукта",
    },
    ws_unmapped: {"Аплифт из Excel, п.п."},
}
for ws, headers_to_format in numeric_format_headers.items():
    header_to_col = {cell.value: cell.column for cell in ws[1]}
    for header in headers_to_format:
        for row_number in range(2, ws.max_row + 1):
            ws.cell(row_number, header_to_col[header]).number_format = "0.0"

for ws in (ws_detail, ws_summary):
    header_to_col = {cell.value: cell.column for cell in ws[1]}
    overstatement_col = header_to_col["Завышение, п.п."]
    letter = get_column_letter(overstatement_col)
    if ws.max_row >= 2:
        ws.conditional_formatting.add(
            f"{letter}2:{letter}{ws.max_row}",
            CellIsRule(operator="greaterThan", formula=["0"], fill=red_fill),
        )

for ws in workbook.worksheets:
    for column_cells in ws.columns:
        letter = get_column_letter(column_cells[0].column)
        max_len = max((len(str(cell.value)) for cell in column_cells if cell.value is not None), default=0)
        ws.column_dimensions[letter].width = min(max(max_len + 2, 10), 36)

ws_detail.column_dimensions["C"].width = 36
ws_detail.column_dimensions["D"].width = 72
ws_detail.column_dimensions["K"].width = 34
ws_detail.column_dimensions["O"].width = 46
ws_detail.column_dimensions["P"].width = 46
ws_summary.column_dimensions["C"].width = 64
ws_summary.column_dimensions["H"].width = 72
ws_summary.column_dimensions["I"].width = 34
ws_unmapped.column_dimensions["C"].width = 72
ws_unmapped.column_dimensions["E"].width = 54
ws_unmapped.column_dimensions["F"].width = 72
ws_method.column_dimensions["A"].width = 36
ws_method.column_dimensions["B"].width = 120

for ws in (ws_detail, ws_summary, ws_unmapped):
    for row_number in range(2, ws.max_row + 1):
        ws.row_dimensions[row_number].height = 54
for row_number in range(2, ws_method.max_row + 1):
    ws_method.row_dimensions[row_number].height = 48

# Реальные Excel tables сохраняют прямоугольный табличный формат и фильтры.
for index, ws in enumerate(workbook.worksheets, start=1):
    table = Table(displayName=f"AuditTable{index}", ref=ws.dimensions)
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium4", showFirstColumn=False, showLastColumn=False,
        showRowStripes=True, showColumnStripes=False,
    )
    ws.add_table(table)

workbook.save(OUTPUT_XLSX)
print(f"Saved {OUTPUT_XLSX_REL} ({OUTPUT_XLSX.stat().st_size:,} bytes)")


Saved artifacts/Аудит_аплифта_дорожных_карт_DD.xlsx (34,749 bytes)


### 8. Reopen and validate workbook structure and mandatory values


In [8]:
formula_view = load_workbook(OUTPUT_XLSX, data_only=False, read_only=False)
value_view = load_workbook(OUTPUT_XLSX, data_only=True, read_only=False)
expected_sheets = ["Ошибки — детализация", "Ошибки — по метрикам", "Не сопоставлено", "Методика"]
assert formula_view.sheetnames == expected_sheets
assert value_view.sheetnames == expected_sheets

detail_ws = formula_view["Ошибки — детализация"]
summary_ws = formula_view["Ошибки — по метрикам"]
unmapped_ws = formula_view["Не сопоставлено"]
assert [cell.value for cell in detail_ws[1]][:7] == DETAIL_HEADERS[:7]
assert detail_ws.max_row - 1 == flagged_detail_metric_rows
assert summary_ws.max_row - 1 == len(flagged_groups)
assert unmapped_ws.max_row - 1 == len(unmatched_numeric_items) + len(overlap_groups)

formula_cells = []
for ws in formula_view.worksheets:
    assert ws.freeze_panes == "A2"
    assert ws.auto_filter.ref == ws.dimensions
    assert len(ws.tables) == 1
    for row in ws.iter_rows():
        for cell in row:
            if isinstance(cell.value, str) and cell.value.startswith("="):
                formula_cells.append((ws.title, cell.coordinate, cell.value))
assert not formula_cells

for reopened in (formula_view, value_view):
    ws = reopened["Ошибки — детализация"]
    headers = {cell.value: cell.column for cell in ws[1]}
    summary_sheet = reopened["Ошибки — по метрикам"]
    summary_headers = {cell.value: cell.column for cell in summary_sheet[1]}
    unmatched_sheet = reopened["Не сопоставлено"]
    unmatched_headers = {cell.value: cell.column for cell in unmatched_sheet[1]}
    generic_alert_source_rows = {11, 17, 21, 25}
    generic_alert_unmatched_rows = [
        row_number for row_number in range(2, unmatched_sheet.max_row + 1)
        if unmatched_sheet.cell(row_number, unmatched_headers["Лист-источник"]).value == "CX"
        and unmatched_sheet.cell(row_number, unmatched_headers["Строка-источник"]).value in generic_alert_source_rows
    ]
    assert len(generic_alert_unmatched_rows) == 4
    assert {
        unmatched_sheet.cell(row_number, unmatched_headers["Строка-источник"]).value
        for row_number in generic_alert_unmatched_rows
    } == generic_alert_source_rows
    for row_number in generic_alert_unmatched_rows:
        assert unmatched_sheet.cell(row_number, unmatched_headers["Планируемое мероприятие"]).value == "Внедрить систему оповещений"
        assert unmatched_sheet.cell(row_number, unmatched_headers["Причина"]).value
        assert unmatched_sheet.cell(row_number, unmatched_headers["Лучшие кандидаты"]).value
    detail_refs = {
        str(ws.cell(row_number, headers["Листы / строки источника"]).value)
        for row_number in range(2, ws.max_row + 1)
    }
    summary_refs = {
        str(summary_sheet.cell(row_number, summary_headers["Листы / строки источника"]).value)
        for row_number in range(2, summary_sheet.max_row + 1)
    }
    assert all(f"CX!{source_row}" not in refs for source_row in generic_alert_source_rows for refs in detail_refs | summary_refs)
    for source_row, (expected_metric, expected_excel, expected_actual) in mandatory_spots.items():
        source_ref = f"УБ!{source_row}"
        rows = [
            row_number for row_number in range(2, ws.max_row + 1)
            if source_ref in str(ws.cell(row_number, headers["Листы / строки источника"]).value)
            and ws.cell(row_number, headers["Продукт"]).value == "Страхование залога"
            and ws.cell(row_number, headers["Метрика"]).value == expected_metric
        ]
        assert len(rows) == 1, (source_row, rows)
        row_number = rows[0]
        assert as_decimal(ws.cell(row_number, headers["Аплифт из Excel, п.п."]).value) == expected_excel
        assert as_decimal(ws.cell(row_number, headers["Настоящий аплифт метрики, п.п."]).value) == expected_actual
        assert as_decimal(ws.cell(row_number, headers["Настоящий аплифт набора метрик, п.п."]).value) == expected_actual
        assert as_decimal(ws.cell(row_number, headers["Настоящий аплифт метрики, п.п."]).value) != Decimal("12.4")

    dsj_rows = [
        row_number for row_number in range(2, summary_sheet.max_row + 1)
        if summary_sheet.cell(row_number, summary_headers["Продукт"]).value == "ДСЖ ПК"
    ]
    assert len(dsj_rows) == 1
    dsj_row_number = dsj_rows[0]
    metric_label = summary_sheet.cell(dsj_row_number, summary_headers["Набор метрик"]).value
    assert "Клиенты с продуктом" in metric_label and "MAU продукта" in metric_label
    assert as_decimal(summary_sheet.cell(dsj_row_number, summary_headers["Excel total группы, п.п."]).value) == Decimal("30.0")
    assert as_decimal(summary_sheet.cell(dsj_row_number, summary_headers["Настоящий аплифт набора метрик, п.п."]).value) == Decimal("1.9")

number_format_checks = []
for header in ("Аплифт из Excel, п.п.", "Настоящий аплифт метрики, п.п.", "Завышение, п.п."):
    col = {cell.value: cell.column for cell in detail_ws[1]}[header]
    if detail_ws.max_row >= 2:
        number_format_checks.append(detail_ws.cell(2, col).number_format)
assert all(fmt == "0.0" for fmt in number_format_checks)
assert len(detail_ws.conditional_formatting) >= 1
assert len(summary_ws.conditional_formatting) >= 1

validation = {
    "status": "PASS",
    "sheets": formula_view.sheetnames,
    "detail_metric_rows": detail_ws.max_row - 1,
    "flagged_metric_set_rows": summary_ws.max_row - 1,
    "unmatched_and_overlap_rows": unmapped_ws.max_row - 1,
    "output_formula_cells": len(formula_cells),
    "excel_tables": sum(len(ws.tables) for ws in formula_view.worksheets),
    "reopened_data_only_false": True,
    "reopened_data_only_true": True,
    "mandatory_spots": spot_results,
    "reviewed_regression_spots": reviewed_regression_results,
    "dsj_pk": "30.0 vs 1.9 on Клиенты с продуктом + MAU продукта",
    "generic_alert_rows_unmatched": 4,
}
print(validation)


{'status': 'PASS', 'sheets': ['Ошибки — детализация', 'Ошибки — по метрикам', 'Не сопоставлено', 'Методика'], 'detail_metric_rows': 65, 'flagged_metric_set_rows': 45, 'unmatched_and_overlap_rows': 31, 'output_formula_cells': 0, 'excel_tables': 4, 'reopened_data_only_false': True, 'reopened_data_only_true': True, 'mandatory_spots': {302: {'metric': 'Регулярность', 'excel_pp': 2.9, 'actual_pp': 0.0, 'inclusion': 'Исключена: excluded_from_index=true; dd_calculation_flg=0'}, 305: {'metric': 'Наличие Self-service', 'excel_pp': 2.9, 'actual_pp': 2.1, 'inclusion': 'Включена'}, 306: {'metric': 'Наличие успешных бизнес-запусков', 'excel_pp': 2.9, 'actual_pp': 2.1, 'inclusion': 'Включена'}}, 'reviewed_regression_spots': {'DP!4': {'metrics': ['Прогноз по целям', 'Факторный анализ - драйверы 1-2 ур.', 'Цели выведены на мониторинг'], 'excel_pp': 10.5, 'actual_pp': 10.6}, 'СВР!7': {'metrics': ['Объем целевого рынка в России', 'Объем целевого рынка в Сбере'], 'excel_pp': 2.4, 'actual_pp': 1.6}, 'УБ!3

## Takeaways

- Error claim возникает только после детерминированного item→metric-set сопоставления и группировки одинаковых наборов.
- Generic/ambiguous items и пересекающиеся неидентичные sets исключены из error claims и перечислены отдельно.
- Исключённые matched metrics видимы в детализации, но их actual uplift равен 0.
- Workbook повторно открывается в formula/value режимах; обязательные reviewed mappings, Страхование залога и ДСЖ ПК spot-checks валидируются assertion-ами.
